In [ ]:
# Cell 1 — Install dependencies
!pip install -q -U google-genai requests==2.32.4

In [ ]:
# Cell 2 — API key setup (key is entered at runtime, never saved in code)
import os

GEMINI_API_KEY = None

# Strategy 1: Google Colab Secrets (desktop only — key icon in left sidebar)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if GEMINI_API_KEY:
        print("API key loaded from Colab Secrets.")
except Exception:
    pass

# Strategy 2: Environment variable
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
    if GEMINI_API_KEY:
        print("API key loaded from environment variable.")

# Strategy 3: Runtime prompt — key is typed when cell runs, never stored in code
if not GEMINI_API_KEY:
    GEMINI_API_KEY = input("Paste your Gemini API key and press Enter: ").strip()
    if GEMINI_API_KEY:
        print("API key accepted. Clear this cell's output to hide it from view.")

if not GEMINI_API_KEY:
    raise ValueError("No API key provided.")

In [ ]:
# Cell 3 — Configure Gemini client (new google-genai SDK)
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client configured.")

In [ ]:
# Cell 4 — Embedder helper + tool schema (new google-genai SDK)
import requests
from google.genai import types

EMBEDDER_URL = "https://feel-dork-entree.ngrok-free.dev/ask"  # update if ngrok restarts

def ask_embedder(query: str) -> str:
    try:
        resp = requests.post(EMBEDDER_URL, json={"question": query}, timeout=10)
        resp.raise_for_status()
        return resp.json().get("answer", "No answer field in response.")
    except requests.exceptions.Timeout:
        return "Error: The embedder API timed out."
    except requests.exceptions.RequestException as e:
        return f"Error calling embedder API: {e}"

pediatric_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="pediatric_fact_lookup",
            description=(
                "Look up a factual answer from the pediatric curriculum knowledge base. "
                "Use this whenever the user asks a question about pediatric medicine, "
                "child health, developmental milestones, vital signs, dosing, or any "
                "topic covered in the pediatric curriculum."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(
                        type=types.Type.STRING,
                        description="The pediatric question to look up."
                    )
                },
                required=["query"]
            )
        )
    ]
)

print("Tool and helper function defined.")

In [ ]:
# Cell 5 — Chat configuration
CHAT_CONFIG = types.GenerateContentConfig(
    tools=[pediatric_tool],
    system_instruction=(
        "You are a helpful pediatric medical education assistant. "
        "You have access to a validated pediatric curriculum knowledge base via the "
        "pediatric_fact_lookup tool. Always use this tool when the user asks a factual "
        "question about pediatrics. Synthesise the retrieved fact into a clear, "
        "conversational answer. If the knowledge base returns no relevant answer, "
        "say so honestly and suggest the user consult a clinical resource."
    )
)

print("Chat configuration ready.")

In [ ]:
# Cell 6 — Chat turn handler
def chat_turn(chat, user_message: str) -> str:
    response = chat.send_message(user_message)

    while True:
        if not response.function_calls:
            return response.text or "(No response)"

        fn_response_parts = []
        for fn_call in response.function_calls:
            name = fn_call.name
            args = dict(fn_call.args)

            if name == "pediatric_fact_lookup":
                result = ask_embedder(args.get("query", ""))
            else:
                result = f"Unknown tool: {name}"

            fn_response_parts.append(
                types.Part.from_function_response(
                    name=name,
                    response={"result": result}
                )
            )

        response = chat.send_message(fn_response_parts)

print("Chat handler ready.")

In [ ]:
# Cell 7 — Conversation loop (run manually; type 'quit' to stop)
chat = client.chats.create(
    model="gemini-2.0-flash",
    config=CHAT_CONFIG
)

print("Pediatric Curriculum Chatbot (gemini-2.0-flash)")
print("Type 'quit' or 'exit' to stop.\n")

while True:
    try:
        user_input = input("You: ").strip()
    except EOFError:
        print("\n[Input stream ended — stopping chatbot]")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    print("Assistant: ", end="", flush=True)
    try:
        reply = chat_turn(chat, user_input)
        print(reply)
    except Exception as e:
        print(f"[Error: {e}]")

    print()